# Rename Workspace Items — Add Prefix

Adds a prefix to all **Reports**, **Paginated Reports**, and **Semantic Models** in a Fabric workspace.

**Use case:** Rename items before making changes so you can easily identify the original versions.

| Step | Description |
|------|-------------|
| 1 | Configure workspace ID, prefix, and credentials |
| 2 | Authenticate |
| 3 | List all items (preview) |
| 4 | Dry-run — show proposed renames |
| 5 | Apply renames |

In [ ]:
# Cell 2 — Configuration

WORKSPACE_ID = "00000000-0000-0000-0000-000000000000"   # Target workspace GUID

PREFIX = "BACKUP_"   # Prefix to add (e.g. "OLD_", "BACKUP_", "v1_")

# Set to True to skip items that already have the prefix
SKIP_ALREADY_PREFIXED = True

# Which item types to rename
RENAME_REPORTS = True
RENAME_PAGINATED_REPORTS = True
RENAME_SEMANTIC_MODELS = True

# Authentication — fill in ONE of the two options below
# Option A: Service Principal
TENANT_ID   = ""  
CLIENT_ID   = ""
CLIENT_SECRET = ""   # or use an env var: os.environ.get("AZURE_CLIENT_SECRET", "")

# Option B: Interactive browser login (leave SP fields blank)
USE_INTERACTIVE = True   # Set False to use SP creds above

In [ ]:
# Cell 3 — Install dependencies (run once)
%pip install requests msal --quiet

In [ ]:
# Cell 4 — Authenticate
import requests, json, os
from msal import ConfidentialClientApplication, PublicClientApplication

SCOPES = ["https://analysis.windows.net/powerbi/api/.default"]

def get_token_sp():
    app = ConfidentialClientApplication(
        CLIENT_ID, authority=f"https://login.microsoftonline.com/{TENANT_ID}",
        client_credential=CLIENT_SECRET)
    result = app.acquire_token_for_client(scopes=SCOPES)
    if "access_token" not in result:
        raise RuntimeError(f"SP auth failed: {result.get('error_description', result)}")
    return result["access_token"]

def get_token_interactive():
    app = PublicClientApplication(
        "04b07795-a71b-4346-935f-02f9a1efa4ce",  # Azure CLI well-known client
        authority="https://login.microsoftonline.com/organizations")
    result = app.acquire_token_interactive(scopes=["https://analysis.windows.net/powerbi/api/Workspace.ReadWrite.All"])
    if "access_token" not in result:
        raise RuntimeError(f"Interactive auth failed: {result.get('error_description', result)}")
    return result["access_token"]

if USE_INTERACTIVE:
    TOKEN = get_token_interactive()
else:
    TOKEN = get_token_sp()

HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
print("\u2713 Authenticated successfully")

In [ ]:
# Cell 5 — Helper: list items from Fabric API
FABRIC_BASE = "https://api.fabric.microsoft.com/v1"
PBI_BASE    = "https://api.powerbi.com/v1.0/myorg"

def list_fabric_items(item_type):
    """List items of a given type via Fabric REST API."""
    url = f"{FABRIC_BASE}/workspaces/{WORKSPACE_ID}/items?type={item_type}"
    items = []
    while url:
        r = requests.get(url, headers=HEADERS, timeout=60)
        r.raise_for_status()
        data = r.json()
        items.extend(data.get("value", []))
        url = data.get("continuationUri")
    return items

def list_pbi_reports():
    """List all reports (incl paginated) via Power BI REST API."""
    url = f"{PBI_BASE}/groups/{WORKSPACE_ID}/reports"
    r = requests.get(url, headers=HEADERS, timeout=60)
    r.raise_for_status()
    return r.json().get("value", [])

def list_pbi_datasets():
    """List semantic models via Power BI REST API."""
    url = f"{PBI_BASE}/groups/{WORKSPACE_ID}/datasets"
    r = requests.get(url, headers=HEADERS, timeout=60)
    r.raise_for_status()
    return r.json().get("value", [])

print("Helpers loaded")

In [ ]:
# Cell 6 — List all items (preview)

reports = list_pbi_reports() if RENAME_REPORTS or RENAME_PAGINATED_REPORTS else []
datasets = list_pbi_datasets() if RENAME_SEMANTIC_MODELS else []

regular_reports = [r for r in reports if r.get("reportType") != "PaginatedReport"]
paginated_reports = [r for r in reports if r.get("reportType") == "PaginatedReport"]

print(f"Reports:            {len(regular_reports)}")
print(f"Paginated Reports:  {len(paginated_reports)}")
print(f"Semantic Models:    {len(datasets)}")
print()

if RENAME_REPORTS:
    print("--- Reports ---")
    for r in regular_reports:
        print(f"  {r['name']}  ({r['id']})")

if RENAME_PAGINATED_REPORTS:
    print("\n--- Paginated Reports ---")
    for r in paginated_reports:
        print(f"  {r['name']}  ({r['id']})")

if RENAME_SEMANTIC_MODELS:
    print("\n--- Semantic Models ---")
    for d in datasets:
        print(f"  {d['name']}  ({d['id']})")

In [ ]:
# Cell 7 — Build rename plan (DRY RUN)

rename_plan = []  # list of (item_type, id, old_name, new_name)

def plan_renames(items, item_type):
    for item in items:
        name = item.get("name") or item.get("displayName", "")
        item_id = item["id"]
        if SKIP_ALREADY_PREFIXED and name.startswith(PREFIX):
            continue
        new_name = f"{PREFIX}{name}"
        rename_plan.append((item_type, item_id, name, new_name))

if RENAME_REPORTS:
    plan_renames(regular_reports, "Report")
if RENAME_PAGINATED_REPORTS:
    plan_renames(paginated_reports, "PaginatedReport")
if RENAME_SEMANTIC_MODELS:
    plan_renames(datasets, "SemanticModel")

print(f"Items to rename: {len(rename_plan)}\n")
print(f"{'Type':<20} {'Current Name':<45} {'New Name'}")
print("-" * 110)
for item_type, item_id, old_name, new_name in rename_plan:
    print(f"{item_type:<20} {old_name:<45} {new_name}")

if not rename_plan:
    print("Nothing to rename (all items may already have the prefix).")

In [ ]:
# Cell 8 — APPLY RENAMES  (run only after reviewing the dry-run above)
import time

def rename_fabric_item(item_id, new_name, item_type):
    """Rename an item via the Fabric Items API (PATCH)."""
    url = f"{FABRIC_BASE}/workspaces/{WORKSPACE_ID}/items/{item_id}"
    payload = {"displayName": new_name}
    r = requests.patch(url, headers=HEADERS, json=payload, timeout=60)
    r.raise_for_status()
    return True

success = 0
failed = 0

for item_type, item_id, old_name, new_name in rename_plan:
    try:
        rename_fabric_item(item_id, new_name, item_type)
        print(f"  \u2713 {item_type}: {old_name} -> {new_name}")
        success += 1
    except requests.exceptions.HTTPError as e:
        print(f"  \u2717 {item_type}: {old_name} — {e.response.status_code}: {e.response.text}")
        failed += 1
    # Small delay to avoid throttling
    time.sleep(0.5)

print(f"\nDone — {success} renamed, {failed} failed")

## Undo — Remove Prefix

Run the cell below to **reverse** the rename by stripping the prefix from all items that currently have it.

In [ ]:
# Cell 10 — UNDO: Remove prefix from all items that have it
import time

# Re-fetch current state
all_reports = list_pbi_reports()
all_datasets = list_pbi_datasets()

undo_plan = []

for r in all_reports:
    rtype = "PaginatedReport" if r.get("reportType") == "PaginatedReport" else "Report"
    if r["name"].startswith(PREFIX):
        undo_plan.append((rtype, r["id"], r["name"], r["name"][len(PREFIX):]))

for d in all_datasets:
    if d["name"].startswith(PREFIX):
        undo_plan.append(("SemanticModel", d["id"], d["name"], d["name"][len(PREFIX):]))

print(f"Items to un-prefix: {len(undo_plan)}\n")
for item_type, item_id, old_name, new_name in undo_plan:
    print(f"  {item_type}: {old_name} -> {new_name}")

if undo_plan:
    confirm = input("\nType YES to proceed: ")
    if confirm.strip() == "YES":
        ok = 0
        for item_type, item_id, old_name, new_name in undo_plan:
            try:
                rename_fabric_item(item_id, new_name, item_type)
                print(f"  \u2713 {old_name} -> {new_name}")
                ok += 1
            except Exception as e:
                print(f"  \u2717 {old_name} — {e}")
            time.sleep(0.5)
        print(f"\nDone — {ok} reverted")
    else:
        print("Cancelled.")